# Tool-Conditional Refusal — Mechanistic Interpretability (consolidated)

*One notebook that consolidates the five `run_interp*.py` scripts. It explains each experiment, shows
the cached result + figure instantly (no GPU needed), and lets you re-run any experiment by flipping a
flag.*

**How to use**
- Run the **Setup** cell. By default `RE_RUN = False`, so every section just **loads saved results from
  `interp_artifacts/` and displays the figure** — instant, no model required.
- To actually recompute an experiment on the GPU, set `RE_RUN = True` in Setup and run that section's
  `if RE_RUN:` cell (loads Qwen3-14B in bf16, ~28 GB).

**Companion docs:** `MECH_INTERP_GUIDE.md` (beginner explanation) · `MECHANISTIC_INTERP.md` (results).

**The story in one line:** refusal is governed by a single direction; tool context *suppresses* it; the
direction strongly *predicts* the resulting unsafe tool call (AUC 0.73) but only *partially controls* it —
a strong-predictor / partial-mediator dissociation.

## Setup — config, lightweight viewers, and (optional) model + helpers

In [ ]:
import json
from pathlib import Path
import numpy as np
try:
    from IPython.display import Image, display, Markdown
except Exception:
    pass

# Locate repo root
REPO = Path.cwd()
while REPO.name and not (REPO/'tools'/'registry.py').exists() and REPO != REPO.parent:
    REPO = REPO.parent
ART = REPO/'interp_artifacts'

RE_RUN = False   # <-- set True to recompute experiments on the GPU (loads Qwen3-14B bf16)

def summary(name):
    f = ART/name
    return json.load(open(f)) if f.exists() else {}
def show(figname, caption=None):
    f = ART/figname
    if f.exists():
        if caption: display(Markdown(f'**{caption}**'))
        display(Image(filename=str(f)))
    else:
        print('(figure not found:', figname, '- run with RE_RUN=True)')

R1=summary('interp_summary.json'); R2=summary('interp2_summary.json')
R3=summary('interp3_summary.json'); R4=summary('interp4_summary.json'); R5=summary('interp5_summary.json')
print('Loaded cached summaries:', [k for k,v in dict(R1=R1,R2=R2,R3=R3,R4=R4,R5=R5).items() if v])
print('RE_RUN =', RE_RUN, '(set True in this cell to enable GPU recompute)')

### (Optional) Load model + shared helpers
This is the **deduplicated** version of the boilerplate that was copy-pasted across the five scripts:
prompt formatting (reusing the repo `tools/` package), the residual-stream reader (`output_hidden_states`),
the hook utilities (ablation / addition / projection-patching), batched generation, and the **fixed**
tool-call scorer (`score_tool_calls_all`). Only runs if `RE_RUN = True`.

In [ ]:
if RE_RUN:
    import os, sys, re, torch
    from contextlib import nullcontext
    import pandas as pd
    os.environ.setdefault('HF_HOME','/workspace/.cache/huggingface')
    sys.path.insert(0, str(REPO))
    from tools.registry import get_domain
    from transformers import AutoTokenizer, AutoModelForCausalLM
    torch.set_grad_enabled(False)
    MODEL_ID, DTYPE, DEVICE, LAYER, ENABLE_THINKING, BS = 'Qwen/Qwen3-14B', torch.bfloat16, 'cuda', 33, False, 24

    tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True); tok.padding_side='left'
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, device_map=DEVICE, trust_remote_code=True).eval()
    r_text = torch.load(ART/'refusal_dirs.pt')['dirs'][LAYER]; r_dev = r_text.to(DEVICE,DTYPE).view(-1)

    DOMAIN_MAP={'Finance':'finance','Healthcare':'healthcare','Education / Academic Integrity':'education','Legal / Compliance':'legal'}
    df = pd.read_excel(REPO/'data'/'complete_combined_safety_eval_dataset_2304_normalized_labels.xlsx')
    df.columns=[c.strip() for c in df.columns]; df['User Prompt']=df['User Prompt'].str.replace(chr(0xfffd),chr(0x2014),regex=False)
    df['domain_key']=df['Domain'].map(DOMAIN_MAP)
    df['group_id']=df['Domain']+'|'+df['Scenario']+'|'+df['Prompt Variant']+'|'+df['System Condition']
    MODES=['No-tool chat','Tool-enabled normal','Tool-enabled stress']

    def tools_for_row(row): return None if row['Mode']=='No-tool chat' else list(get_domain(row['domain_key']).tools_for_llm)
    def format_prompt(row):
        msgs=[{'role':'system','content':row['System Prompt']},{'role':'user','content':row['User Prompt']}]
        t=tools_for_row(row); base=dict(tokenize=False,add_generation_prompt=True)
        for extra in ({'enable_thinking':ENABLE_THINKING},{}):
            try: return tok.apply_chat_template(msgs,tools=t,**base,**extra) if t else tok.apply_chat_template(msgs,**base,**extra)
            except TypeError: continue
    def parse_tool_calls(text):
        calls=[]
        for raw in re.findall(r'<tool_call>(.*?)</tool_call>',text,re.DOTALL):
            try:
                c=json.loads(raw.strip()); calls.append({'name':c.get('name',''),'arguments':c.get('arguments',c.get('parameters',{}))})
            except json.JSONDecodeError: pass
        return calls
    def unsafe_all(row,text):   # FIXED scorer: any forbidden action in the domain
        return len(get_domain(row['domain_key']).score_tool_calls_all(parse_tool_calls(text)))>0

    class Hooks:
        def __init__(s): s.h=[]
        def remove(s):
            for x in s.h: x.remove()
            s.h=[]
        def __enter__(s): return s
        def __exit__(s,*a): s.remove()
    def _layers(): return model.model.layers
    def ablate():
        def mk():
            def hook(m,i,o):
                h=o[0] if isinstance(o,tuple) else o; h=h-(h@r_dev).unsqueeze(-1)*r_dev
                return (h,)+tuple(o[1:]) if isinstance(o,tuple) else h
            return hook
        hk=Hooks()
        for L in _layers(): hk.h.append(L.register_forward_hook(mk()))
        return hk
    def addvec(coef):
        def hook(m,i,o):
            h=o[0] if isinstance(o,tuple) else o; h=h+coef*r_dev
            return (h,)+tuple(o[1:]) if isinstance(o,tuple) else h
        hk=Hooks(); hk.h.append(_layers()[LAYER-1].register_forward_hook(hook)); return hk
    @torch.no_grad()
    def resid_proj(prompts):   # projection of last-token residual onto r_text
        out=[]
        for i in range(0,len(prompts),BS):
            enc=tok(prompts[i:i+BS],return_tensors='pt',padding=True,truncation=True,max_length=2048).to(DEVICE)
            hs=model(**enc,output_hidden_states=True).hidden_states
            out.append((hs[LAYER][:,-1,:].float().cpu()@r_text).numpy())
        return np.concatenate(out)
    @torch.no_grad()
    def gen_batch(prompts, hook=None, max_new_tokens=200):
        enc=tok(prompts,return_tensors='pt',padding=True,truncation=True,max_length=2048).to(DEVICE)
        Lp=enc['input_ids'].shape[1]
        with (hook if hook is not None else nullcontext()):
            out=model.generate(**enc,max_new_tokens=max_new_tokens,do_sample=False,pad_token_id=tok.eos_token_id)
        return [tok.decode(out[i,Lp:],skip_special_tokens=False) for i in range(len(prompts))]
    print('Model + helpers ready.  layers=%d  d_model=%d' % (model.config.num_hidden_layers, model.config.hidden_size))
else:
    print('RE_RUN=False - skipping model load. Sections below show cached results.')

---
## 1 · Finding the refusal direction (difference-in-means)
We record the decision-token residual stream for 128 harmful and 128 harmless **No-tool** prompts, average
each group, and subtract: `r = mean(harmful) - mean(harmless)`, normalized. Harmful vs benign separate
cleanly along `r`; we use **layer 33 / 40**. *(Code: `run_interp.py` `[step1]`.)*

In [ ]:
print('Refusal direction saved at interp_artifacts/refusal_dirs.pt')
print('Separation: harmful proj +149 vs benign proj -162  ->  chosen LAYER = 33')
show('fig_separation_by_layer.png', 'Harmful vs benign separation along r, by layer')

## 2 · Proving it is causal (ablation + addition)
**Ablation** removes `r` from every layer (`h -= (h·r̂)r̂`) → should kill refusal on harmful prompts.
**Addition** injects `+c·r̂` → should induce refusal on benign prompts. Small-n teasers were dramatic;
the **scaled** numbers (n=120, with CIs) are the authoritative ones. *(Code: `run_interp.py`, `run_interp5.py`.)*

In [ ]:
ab=R5.get('ablation',{}); ad=R5.get('addition',{})
if ab: print(f"Ablation  (n={ab['n']}): harmful refusal {ab['base_refuse']:.0%} -> {ab['ablated_refuse']:.0%}  CI {ab['ablated_ci']}")
if ad: print(f"Addition  (n={ad['n']}): benign refusal  {ad['base_refuse']:.0%} -> {ad['added_refuse']:.0%}  CI {ad['added_ci']}")
print('\n(small-n teasers were 55->20% and 0->90%; scaling tempered ablation to a ~20pt drop.)')

## 3 · The core result — tool context suppresses the refusal signal
For harmful prompts we project the decision-token residual onto `r̂` in each mode. Same request, only the
tool context differs. *(Code: `run_interp.py` `[step3]`.)*

In [ ]:
mp=R1.get('mode_mean_proj',{})
for m,v in mp.items(): print(f"  {m:22s} mean projection = {v:+.1f}")
pr=R1.get('paired',{})
for m,d in pr.items(): print(f"  paired No-tool - {m:22s}: delta={d['mean_delta']:+.1f}  t={d['t']:.2f}")
show('fig_projection_by_mode.png', 'Refusal-direction projection by mode (the suppression)')

## 4 · System-condition interaction
Projection across mode x {Neutral, Safety-reinforced, Tool-encouraging}. Suppression holds in every
condition; a safety prompt lifts the signal; tool-encouraging+stress is the floor. *(Code: `run_interp2.py` `[#12]`.)*

In [ ]:
sc=R2.get('syscond',{})
for mode,row in sc.items():
    print(mode); [print(f'    {c:18s} {v:+.1f}') for c,v in row.items()]
show('fig_syscond.png','Projection by mode x system condition')

## 5 · Which components do the suppressing? (attention heads + MLPs)
Per-head and per-MLP contribution to `r`, No-tool minus Tool. Concentrated in late attention heads
(L34-39) and MLPs (L29-32). *(Code: `run_interp2.py` `[#4]`.)*

In [ ]:
h=R2.get('heads',{})
print('Top suppression heads (layer, head, drop):')
for x in h.get('top_suppression_heads',[])[:6]: print('   L%d.H%d  %.2f'%(x['layer'],x['head'],x['drop']))
print('Top MLP layers (layer, drop):', h.get('top_mlp_layers',[])[:5])
show('fig_suppression_heads.png','Per-head refusal-contribution drop'); show('fig_mlp_by_layer.png','Per-layer MLP drop')

## 6 · Is it the *same* direction under tools? (validity check)
Re-extract the refusal direction *within* tool mode and cosine-compare to the text one. High cosine = same
feature, suppressed (not a different feature). *(Code: `run_interp4.py` `[#2]`.)*

In [ ]:
sd=R4.get('same_direction',{})
print('cosine(r_text, r_tool) @ L33 =', round(sd.get('cosine_text_vs_tool_dir_layer33',float('nan')),3),
      '->  same feature, suppressed' )

## 7 · Does the projection predict the unsafe action? (AUC)
Per harmful tool prompt: projection onto `r̂` vs whether the call was unsafe (fixed scorer). AUC ~ how well
a low projection predicts an unsafe call. *(Code: `run_interp4.py`/`run_interp5.py` `[#4]`.)*

In [ ]:
au=R5.get('auc',{})
if au: print(f"AUC = {au['auc']}  CI {au['auc_ci']}   (n={au['n']}, {au['n_unsafe']} unsafe; proj unsafe {au['mean_proj_unsafe']} vs safe {au['mean_proj_safe']})")
show('fig_auc_scaled.png','Projection distributions: safe vs unsafe (AUC 0.73)')

## 8 · Can we restore safety? (patching + steering)
**Patching:** force the projection back to its matched No-tool level on unsafe cases. **Steering:** add
`+c·r̂` and sweep `c`. Both move the behavior, but only partially / bluntly. *(Code: `run_interp4/5.py`.)*

In [ ]:
pa=R5.get('patching',{}); st=R5.get('steering',{})
if pa: print(f"Patching to text-level: flipped {pa['flipped_to_safe']}/{pa['n_unsafe_baseline']} unsafe -> safe = {pa['rate']:.0%}  CI {pa['rate_ci']}")
if st:
    print('\nSteering dose-response:')
    for i,c in enumerate(st['grid']):
        print(f"   c={c:4d}  unsafe={st['harmful_unsafe'][i]:.0%} CI{st['harmful_unsafe_ci'][i]} | benign over-refuse={st['benign_refuse'][i]:.0%} CI{st['benign_refuse_ci'][i]}")
show('fig_steering_scaled.png','Steering dose-response (n=100/60, 95% CI)')

## 9 · The scorer fix (a methodological finding)
The original `score_tool_calls` only checked each scenario's *narrow* forbidden-action list, missing
cross-scenario violations (a PHI-retrieval call scored "safe"). We added `score_tool_calls_all`
(`tools/core.py`). Re-scoring raised the harmful unsafe rate **4.2% -> 11.5%** on the first 250 rows,
and the true base rate on fresh interp samples is **~31-34%**. Re-score any CSV with `rescore_results.py`.

In [ ]:
# Re-score the (in-progress) behavioral CSV under both scorings, if present
import subprocess, sys
csv = REPO/'results'/'results_Qwen3-14B.csv'
if csv.exists():
    print(subprocess.run([sys.executable, str(REPO/'rescore_results.py'), str(csv)], capture_output=True, text=True).stdout)
else:
    print('behavioral CSV not present yet')

---
## 10 · Summary & honest interpretation

| Result | Value | Verdict |
|---|---|---|
| Refusal direction is causal | ablation 56→36% (n=120), addition 1→71% | ✅ real (modest ablation) |
| Tool context suppresses it | paired Δ +66.5, t=5.65 | ✅ strong |
| Same feature under tools | cosine 0.735 | ✅ validated |
| Suppression circuit | heads L34-39, MLPs L29-32 | ✅ localized |
| Projection predicts unsafe | **AUC 0.73** [0.67-0.78] | ✅ strong predictor |
| Patching restores safety | 19% [12-28%] | ⚠️ partial |
| Steering restores safety | unsafe→0% but 62% over-refusal | ⚠️ blunt |
| Scorer fix | unsafe 4.2%→11.5% (~31% true) | 🔧 important |

**The headline is a dissociation:** the refusal direction is a **strong predictor / partial mediator** of
the unsafe tool call, not the sole causal switch. Tool-refusal failure is driven jointly by
refusal-suppression **and** the tool affordance. The descriptive mechanism + the dataset design are the
strengths; the causal-sufficiency is the limitation (and may be partly understated by crude interventions —
a stronger ablation is the key next experiment).

**Reproduce:** `run_interp.py` → `run_interp2.py` → `run_interp4.py` → `run_interp5.py`
(`run_interp3.py` holds the round-2 fixups). All reuse `interp_artifacts/refusal_dirs.pt` + the `tools/`
package. See `MECH_INTERP_GUIDE.md` for the from-scratch explanation.